# Assessing Accuracy of CFF Global Fits:

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Native Libraries:

In [ ]:
import datetime

### (1.2): Import 3rd Party Libraries:

In [ ]:
import numpy as np
from scipy import integrate
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import gepard as g
from gepard.fits import th_KM15

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")

### (1.4): Customizing Plotting Settings:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (2): Data Formatting/Collection Settings:

### (2.1): Versioning:

In [ ]:
VERSION_NUMBER = 1
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

In [ ]:
K_RANGE = np.linspace(5., 11., 13)
Q2_RANGE = np.linspace(1., 5., 17)
X_B_RANGE = np.linspace(0.1, 0.9, 17)
T_RANGE = np.linspace(-1.0, -0.1, 217)

In [ ]:
nval = 1.35
pval = 1.
nsea = 1.5
rsea = 1.
psea = 2.
bsea = 4.6
Mval = 0.789
rval = 0.918
bval = 0.4
C0 = 2.768
Msub = 1.204
Mtval = 3.993
rtval = 0.881
btval = 0.4
ntval = 0.6
Msea = np.sqrt(0.482)
rpi = 2.646
Mpi = 4.

@np.vectorize
# https://stackoverflow.com/a/77409936 -> for how to integrate over meshgrid domain
def compute_km15_cffs(q_squared, xb, t, k = 0.0):
    xi = xb / (2.0 - xb)
    alpha_val = 0.43 + 0.85 * t
    alpha_sea = 1.13 + 0.15 * t
    Ct = C0 / (1.0 - t / Msub**2)**2

    def fHval(x):
        return (nval * rval / (1 + x) *
                ((2 * x) / (1 + x))**(-alpha_val) *
                ((1 - x) / (1 + x))**bval /
                (1 - ((1 - x) / (1 + x)) * (t / Mval**2))**pval)

    def fHsea(x):
        return (nsea * rsea / (1 + x) *
                ((2 * x) / (1 + x))**(-alpha_sea) *
                ((1 - x) / (1 + x))**bsea /
                (1 - ((1 - x) / (1 + x)) * (t / Msea**2))**psea)

    def fImH(x):
        return np.pi * ((8. / 9.) * fHval(x) + (1. / 9.) * fHsea(x))

    def fPV_ReH(x):
        return -2. * x / (x + xi) * fImH(x)
    
    DR_ReH, _ = integrate.quad(fPV_ReH, 1e-6, 1.0, weight = 'cauchy', wvar = xi)

    real_h_km15 = DR_ReH / np.pi - Ct # Re[H]
    imag_h_km15 = fImH(xi) # Im[H]

    return real_h_km15, imag_h_km15

In [ ]:
scrubbed_xb_meshgrid, scrubbed_t_meshgrid = np.meshgrid(X_B_RANGE, T_RANGE, indexing = "ij")

_FIXED_Q_SQUARED_VALUE = 1.0

cff_real_h_const_t, cff_imag_h_const_t = compute_km15_cffs(
    q_squared = _FIXED_Q_SQUARED_VALUE, 
    xb = scrubbed_xb_meshgrid, 
    t = scrubbed_t_meshgrid, 
    k = 0)

In [ ]:
def random_surface(x, y):
    return 2.* (np.random.rand() * x**2 + np.random.rand() * y**2) * np.exp(-1. * np.random.rand() * x * y )

cff_h_real_global_fit_residual = np.abs(cff_real_h_const_t - random_surface(scrubbed_xb_meshgrid, scrubbed_t_meshgrid))
cff_h_imag_global_fit_residual = np.abs(cff_imag_h_const_t - random_surface(scrubbed_xb_meshgrid, scrubbed_t_meshgrid))

## Re[$\mathcal{H}$]$(x_{\textrm{B}}, t, Q^{2} = ??)$ Surface Residual Plot:

In [ ]:
#############################################
# figure initialization and customization
#############################################
real_h_residual_plot_figure = plt.figure()
real_h_residual_plot_figure.set_figheight(8)
real_h_residual_plot_figure.set_figwidth(8)

real_h_residual_plot_axis = real_h_residual_plot_figure.add_subplot(projection = '3d')

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = real_h_residual_plot_axis.elev # extract eleveation param
axis_azimuthal = real_h_residual_plot_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
real_h_residual_plot_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = real_h_residual_plot_axis.transAxes)
real_h_residual_plot_axis.text2D(
    0.01, 0.00, 
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = real_h_residual_plot_axis.transAxes)

# Plot the surface.
real_h_residual_plot_axis.plot_surface(
    scrubbed_xb_meshgrid, scrubbed_t_meshgrid, cff_h_real_global_fit_residual,
    cmap = cm.gray, linewidth = 0, antialiased = False)

real_h_residual_plot_axis.set_xlabel(r"$x_{\textrm{B}}$")
real_h_residual_plot_axis.set_ylabel(r"$t$")
real_h_residual_plot_axis.set_title(
    rf"Residuals for Re[$\mathcal{{H}}$] $\left( x_{{\textrm{{B}}}}, t, Q^{{2}} = {_FIXED_Q_SQUARED_VALUE} \ \textrm{{GeV}}^{{2}} \right ) $ (Model vs. True)")

## Im [$\mathcal{H}$]$(x_{\textrm{B}}, t, Q^{2} = ??)$ Surface Residual Plot:

In [ ]:
#############################################
# figure initialization and customization
#############################################
imag_h_residual_plot_figure = plt.figure()
imag_h_residual_plot_figure.set_figheight(8)
imag_h_residual_plot_figure.set_figwidth(8)

imag_h_residual_plot_axis = imag_h_residual_plot_figure.add_subplot(projection = '3d')

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = imag_h_residual_plot_axis.elev # extract eleveation param
axis_azimuthal = imag_h_residual_plot_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
imag_h_residual_plot_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = imag_h_residual_plot_axis.transAxes)
imag_h_residual_plot_axis.text2D(
    0.01, 0.00, 
    fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = imag_h_residual_plot_axis.transAxes)

# Plot the surface.
imag_h_residual_plot_axis.plot_surface(
    scrubbed_xb_meshgrid, scrubbed_t_meshgrid, cff_h_imag_global_fit_residual,
    cmap = cm.gray, linewidth = 0, antialiased = False)

imag_h_residual_plot_axis.set_xlabel(r"$x_{\textrm{B}}$")
imag_h_residual_plot_axis.set_ylabel(r"$t$")
imag_h_residual_plot_axis.set_title(
    rf"Residuals for Im[$\mathcal{{H}}$] $\left( x_{{\textrm{{B}}}}, t, Q^{{2}} = {_FIXED_Q_SQUARED_VALUE} \ \textrm{{GeV}}^{{2}} \right ) $ (Model vs. True)")